In [37]:
### Kenyan Parliamentary Hansard Analysis Script

## Nicolas de Zamaroczy

## The purpose of this Python script is to perform simple quantitative text analysis of the parsed Kenyan parliamentary hansards.
# In particular, we track sentiment analysis in a variety of ways:
# - over time;
# - by speaker; and
# - by political party.

# The current goal is to see if parliamentary speech is a leading or lagging indicator of political violence in Kenya.

In [38]:
### PRELIMINARIES

# --- Dependency Checks ---
# This block checks if several required libraries are installed.
# If not, it uses Python's package manager (pip) to install them.
# This makes the script more portable and easier for others to run.
try:
    import numpy as np # NumPy, a core package used to structure the data
except ImportError:
    import subprocess
    import sys
    print("NumPy not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])
    import numpy as np
    print("Installation complete.")

try:
    import pandas as pd # Pandas, a core package used to structure the data
except ImportError:
    import subprocess
    import sys
    print("Pandas not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd
    print("Installation complete.")

try:
    from pathlib import Path # PathLib, used to read/write files to a given folder
except ImportError:
    import subprocess
    import sys
    print("PathLib not found. Installing...")
    # It's good to install the optional dependency for faster performance.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pathlib[Path]"])
    from pathlib import Path
    print("Installation complete.")

try:
    from tqdm import tqdm # tqdm, used to create a progress bar
except ImportError:
    import subprocess
    import sys
    print("tqdm not found. Installing...")
    # It's good to install the optional dependency for faster performance.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm[tqdm]"])
    from tqdm import tqdm
    print("Installation complete.")

# Enable tqdm for Pandas
tqdm.pandas()

# Disable Future Deprecation Warnings from getting logged
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

In [39]:
# Load the csv with parsed speech blocks

# Current notebook directory
notebook_dir = Path.cwd()
# Path to your CSV
file_path = notebook_dir.parent / "outputs" / "senate_parsed_outputs.csv"
#file_path = notebook_dir.parent / "outputs" / "national_assembly_parsed_outputs.csv"
# Check existence and raise an error if missing
if not file_path.exists():
    raise FileNotFoundError(f"CSV file not found: {file_path}")

# Read the CSV
df_blocks = pd.read_csv(file_path)

# Remove any \n artifacts that are hidden embeds in the content field
df_blocks["content"] = df_blocks["content"].str.replace(r"\n", " ", regex=True).str.replace(r"\r", " ", regex=True)

# Drop some fields that aren't relevant to the analysis
df_blocks = df_blocks[['filename','date','speaker','matched_name','language_distribution','content','is_continuation']]

# Ensure date column is a datetime
# We're currently getting some messy data in the date field, need to check what's causing it upstream
df_blocks["date"] = pd.to_datetime(df_blocks["date"], errors="coerce")
df_blocks = df_blocks.dropna(subset=["date"])

df_blocks

,filename,date,speaker,matched_name,language_distribution,content,is_continuation
0,25th_November_2015.pdf,2015-11-25,The Speaker (Hon. Ethuro),NaN,"{""en"": 1.0}","Hon. Senators, let us begin with the statement...",False
1,25th_November_2015.pdf,2015-11-25,Sen. Haji,NaN,"{""en"": 1.0}","Mr. Speaker, Sir, the statement is not ready. ...",False
2,25th_November_2015.pdf,2015-11-25,The Speaker (Hon. Ethuro),NaN,"{""en"": 1.0}","What do you have to say, Sen. Mositet?",False
3,25th_November_2015.pdf,2015-11-25,Sen. Mositet,NaN,"{""sw"": 1.0}","Bw. Spika, jana Naibu wa Mwenyeketi alitueleza...",False
4,25th_November_2015.pdf,2015-11-25,The Speaker (Hon. Ethuro),NaN,"{""en"": 1.0}","Order, Senator! The statement will be issued o...",False
...,...,...,...,...,...,...,...
209978,Wednesday__3rd_December_2014.pdf,2014-12-03,Sen. Ndiema,NaN,"{""en"": 1.0}","Thank you, Madam Temporary Speaker. I am seeki...",False
209979,Wednesday__3rd_December_2014.pdf,2014-12-03,The Temporary Speaker (Sen. Ongoro),NaN,"{""en"": 1.0}","Senator, how did you know and yet you do not h...",False
209980,Wednesday__3rd_December_2014.pdf,2014-12-03,Sen. Ndiema,NaN,"{""en"": 1.0}","Madam Temporary Speaker, I do not know whether...",False
209981,Wednesday__3rd_December_2014.pdf,2014-12-03,The Temporary Speaker (Sen. Ongoro),NaN,"{""en"": 1.0}","Unfortunately, you did not manage to conclude ...",False


In [ ]:
# ==============================================================================
# --- CORRELATING TO POLITICAL VIOLENCE ---
# ==============================================================================

### Get ADM-1 Boundaries for Kenya

import geopandas as gpd
import matplotlib.pyplot as plt

# Path to the shapefile (only need to reference the .shp file)
shapefile_path = notebook_dir.parent / "inputs" / "other_data" / "kenya_adm1_shapefiles" / "ken_adm1.shp"
# Check existence and raise an error if missing
if not shapefile_path.exists():
    raise FileNotFoundError(f"CSV file not found: {shapefile_path}")

# Read the shapefile into a GeoDataFrame
gdf = gpd.read_file(shapefile_path)

# Display basic info
print(gdf.head())
print(gdf.crs)

# Simple plot
gdf.plot(edgecolor="black", figsize=(8, 8))
plt.title("Kenya ADM1 Boundaries")
plt.show()

In [ ]:
### Get ACLED Data for Kenya


## There's two ways to do this.
## To get the most up-to-date data, we can hit ACLED's API and pull all Kenyan political violence up to ## days ago (a built-in lag by ACLED).
## Alternatively, if we want to go faster or the API pull isn't working for some reason, we just use the last stored CSV.

# Connecting to ACLED's API

import requests
import json

# Function to get access token using username and password
def get_access_token(username, password, token_url):
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
    }
    data = {
          'username': username,
          'password': password,
          'grant_type': "password",
          'client_id': "acled"
    }

    response = requests.post(token_url, headers=headers, data=data)

    if response.status_code == 200:
        token_data = response.json()
        return token_data['access_token']
    else:
        raise Exception(f"Failed to get access token: {response.status_code} {response.text}")


# Get an access token
# Credentials are read from environment variables (or a local, gitignored .env file).
# See .env.example in the project root for the expected variable names.
import os
try:
    from dotenv import load_dotenv
    load_dotenv(notebook_dir.parent / ".env")
except ImportError:
    pass

my_token = get_access_token(
    username=os.environ["ACLED_USERNAME"],
    password=os.environ["ACLED_PASSWORD"],
    token_url="https://acleddata.com/oauth/token",
)


# (parameters in the url)
base_url = "https://acleddata.com/api/acled/read?_format=csv&country=Georgia"

# request base url with my_token
response = requests.get(
    base_url,
    headers={"Authorization": f"Bearer {my_token}", "Content-Type": "application/json"},
)

if response.json()["status"] == 200:
    print(
        "Request successful"
    )


# # Send parameters (as a dictionary) for the data request
# parameters = {
#     "country": "Georgia:OR:country=Armenia:OR:country=Azerbaijan",
#     "year": 2021,
#     "fields": "event_id_cnty|event_date|event_type|country|fatalities",
# }
# 
# response_params_dic = requests.get(
#     "https://acleddata.com/api/acled/read?_format=csv",
#     params=parameters,
#     headers={"Authorization": f"Bearer {my_token}", "Content-Type": "application/json"},
# )
# if response_params_dic.json()["status"] == 200:
#     print(
#         "Request successful"
#     )

In [3]:
## Load raw data from CSV

# All ACLED incidents in Kenya since October 1st, 2022
df_incidents = pd.read_csv(notebook_dir.parent / "inputs" / "other_data" / "ACLED Kenya Data through 10-17-2024.csv")
    
# Clean data
df_incidents.columns = df_incidents.columns.str.lower()

# Drop strategic developments
#df_incidents.event_type.unique()
df_incidents = df_incidents[df_incidents['event_type']!='Strategic developments'].copy()

df_incidents

NameError: name 'pd' is not defined

In [ ]:
## Plotting a country's events (with ADM-1 boundaries)

## Note: If a country doesn't have a given type of event in the period in question, you need to hash it out.

## Create an empty figure of the desired size
fig, ax = plt.subplots(figsize=(10,10))

## Specify some parameters
title = 'Instability Events in Kenya (by type)'

## Creating the events plots
# Get the data
df_kenya = df_incidents[['event_type','latitude','longitude','admin1','admin2','admin3','location']].copy()

# Convert df_country_events to a Geopandas df
df_geo = gpd.GeoDataFrame(
    df_kenya
    ,geometry=gpd.points_from_xy(df_kenya.longitude, df_kenya.latitude)
    ,crs="epsg:4326" # We specify that we want to use the same CRS as that which the basemap comes in
)
#df_geo

# Create a plot for Protest events
df_geo[df_geo['event_type'] == 'Protests'].plot(ax=ax, 
                                       markersize=12, 
                                       color='green', 
                                       marker='o', 
                                       label='Protests')

# Create a plot for Riot events
df_geo[df_geo['event_type'] == 'Riots'].plot(ax=ax, 
                                       markersize=12, 
                                       color='orange', 
                                       marker='o', 
                                       label='Riots')

# Create a plot for Battle events
df_geo[df_geo['event_type'] == 'Battles'].plot(ax=ax, 
                                       markersize=12, 
                                       color='red', 
                                       marker='x', 
                                       label='Battles')

# Create a plot for Explosion events
df_geo[df_geo['event_type'] == 'Explosions/Remote violence'].plot(ax=ax, 
                                       markersize=12, 
                                       color='darkred', 
                                       marker='*', 
                                       label='Explosions')

# Create a plot for Violence against civilians events
df_geo[df_geo['event_type'] == 'Violence against civilians'].plot(ax=ax, 
                                       markersize=12, 
                                       color='darkred', 
                                       marker='^',
                                       label='Violence against civilians')

## Create a basemap of the country

# Download the shapeid geometries for the country
df_basemap = gpd.read_file(shapefile_path)

# Create the basemap plot
df_basemap["geometry"].boundary.plot(ax=ax) # If you want just an outline

## Add a scale bar (trying to fololow https://geopandas.org/en/stable/gallery/matplotlib_scalebar.html#
## and utterly failing)
#from matplotlib_scalebar.scalebar import ScaleBar
#
# Do math to figure out the distance between two points in this projection
#from shapely.geometry.point import Point
#points = gpd.GeoSeries(
#    [Point(-73.5, 40.5), Point(-74.5, 40.5)], crs=4326
#)  
#points = points.to_crs(epsg=4326)
#distance_meters = points[0].distance(points[1])
#ax.add_artist(ScaleBar(distance_meters,'km'))

# Add tile and legend
plt.title(title)
ax.legend()

# Turn off the axes, since they don't contain useful info
ax.axis('off')

# Export
#plt.savefig(f"C:\\Users\\USER\\Documents\\geopolitical-2.0\\outputs\\plots\\{title}.png".format(title=title))

## Show everything
# Show all created axes simultaneously.
# If you don't do this all in the same cell, the figure resets in the next cell.
plt.show()